In [ ]:
import pandas as pd
import requests
import openpyxl
from io import BytesIO
from google.colab import data_table

def get_automated_timetable(url):
    response = requests.get(url)
    wb = openpyxl.load_workbook(BytesIO(response.content), data_only=True)
    sheet = wb.active

    start_row = None
    end_row = None

    for r in range(1, sheet.max_row + 1):
        val_a = str(sheet.cell(row=r, column=1).value).strip() if sheet.cell(row=r, column=1).value else ""
        val_b = str(sheet.cell(row=r, column=2).value).strip() if sheet.cell(row=r, column=2).value else ""

        if val_a.lower() == "time" or val_b.lower() == "time":
            if start_row is None: start_row = r
        if val_a.upper() == "FRI" or val_b.upper() == "FRI":
            if end_row is None:
                end_row = r
                break

    extracted_data = []
    for r in range(start_row, end_row + 1):
        row_data = []
        for c in range(1, sheet.max_column + 1):
            cell = sheet.cell(row=r, column=c)
            value = cell.value
            for merged_range in sheet.merged_cells.ranges:
                if cell.coordinate in merged_range:
                    value = sheet.cell(row=merged_range.min_row, column=merged_range.min_col).value
                    break

            if value is not None and str(value).strip() != "":
                row_data.append(str(value).strip().replace('\n', ' '))
            else:
                row_data.append("-")
        extracted_data.append(row_data)

    col_letters = [openpyxl.utils.get_column_letter(i) for i in range(1, sheet.max_column + 1)]
    row_numbers = [i for i in range(start_row, end_row + 1)]
    df = pd.DataFrame(extracted_data, columns=col_letters, index=row_numbers)

    # Remove Column A
    if 'A' in df.columns:
        df = df.drop(columns=['A'])

    return df

# --- EXECUTION ---
url = "https://docs.google.com/spreadsheets/*/******************************/export?format=xlsx"
df_final = get_automated_timetable(url)

# Configure Colab to show all columns and avoid truncation
data_table.max_columns = 30
data_table.enable_dataframe_formatter()

df_final

,B,C,D,E,F,G,H,I,J,K,L,M,N,O,P,Q,R,S,T,U
6,Time,07:50-8:50,09:00-10:30,09:00-10:30,10:30-10:45,10:45-11:00,11:00-12:00,12:00-12:15,12:15-12:30,12:30-13:15,13:15-14:00,14:00-15:30,14:00-15:30,15:30-15:40,15:40-17:10,15:40-17:10,15:40-17:10,17:10-17:30,17:30-18:30,18:30 onwards
7,Day,Minor Slot,09:00-10:30,09:00-10:30,-,-,-,-,-,-,-,14:00-15:30,14:00-15:30,-,15:40-17:10,15:40-17:10,15:40-17:10,-,-,Minor Slot
8,MON,-,-,-,-,-,-,-,Data Security and Privacy-T,Data Security and Privacy-T,-,Basket 1,Basket 1,-,-,-,-,-,Basket 3-T,-
9,TUE,-,Basket 5,Basket 5,-,-,-,-,-,-,-,Basket 2,Basket 2,-,Basket 1,Basket 1,Basket 1,-,-,-
10,WED,-,Basket 5,Basket 5,-,Data Security and Privacy,Data Security and Privacy,Data Security and Privacy,-,-,-,Basket 4,Basket 4,-,Basket 3,Basket 3,Basket 3,-,Basket 1-T,-
11,THU,-,Data Security and Privacy,Data Security and Privacy,-,-,-,-,Basket 5-T,Basket 5-T,-,Basket 3,Basket 3,-,Basket 4,Basket 4,Basket 4,-,Basket 2-T,-
12,THU,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
13,FRI,-,-,-,-,-,-,-,-,-,-,-,-,-,Basket 2,Basket 2,Basket 2,-,Basket 4-T,-


In [31]:
from datetime import datetime, timedelta
import re

def timetable_assistant(query):
    query = query.lower().strip()
    time_map = df_final.iloc[0].to_dict()

    # --- INTELLIGENT PARSING ---
    target_date = datetime.now()
    is_date_query = False
    specific_time_filter = None

    # 1. Detect relative days
    if "tomorrow" in query:
        target_date = datetime.now() + timedelta(days=1)
        is_date_query = True
    elif "today" in query:
        target_date = datetime.now()
        is_date_query = True

    # 2. Detect specific dates (e.g., 06-05-2026)
    date_match = re.search(r'(\d{1,2})-(\d{1,2})-(\d{4})', query)
    if date_match:
        target_date = datetime.strptime(date_match.group(), '%d-%m-%Y')
        is_date_query = True

    # 3. Detect specific time mentions (e.g., 11:00 or 12:15)
    time_match = re.search(r'(\d{1,2}:\d{2})', query)
    if time_match:
        specific_time_filter = time_match.group()

    def merge_consecutive_slots(slots):
        if not slots: return []
        merged = []
        current_subject = slots[0][1]
        current_start = slots[0][0].split('-')[0].strip()
        current_end = slots[0][0].split('-')[-1].strip()

        for i in range(1, len(slots)):
            time_range, subject = slots[i]
            slot_start, slot_end = time_range.split('-')[0].strip(), time_range.split('-')[-1].strip()
            if subject == current_subject:
                current_end = slot_end
            else:
                merged.append((current_start, current_end, current_subject))
                current_subject, current_start, current_end = subject, slot_start, slot_end
        merged.append((current_start, current_end, current_subject))
        return merged

    # --- EXECUTION LOGIC ---
    if is_date_query or specific_time_filter:
        day_short = target_date.strftime("%a").upper()
        # Edge case: Sheet might use 'THU' instead of 'THURS' etc.
        day_row = df_final[df_final['B'].str.contains(day_short, na=False, case=False)]

        print(f"--- Schedule for {target_date.strftime('%A, %d %B %Y')} ---")

        if not day_row.empty:
            raw_slots = [(time_map[c], v) for c, v in day_row.iloc[0].items() if c != 'B' and v != "-" and v != "Time"]
            merged = merge_consecutive_slots(raw_slots)

            found = False
            for start, end, subject in merged:
                # If user asked for a specific time, check if that time falls within the class range
                if specific_time_filter:
                    if start <= specific_time_filter < end:
                        print(f"• {start}-{end}: {subject}")
                        found = True
                else:
                    print(f"• {start}-{end}: {subject}")
                    found = True

            if not found: print(f"No classes found for the specific time {specific_time_filter}." if specific_time_filter else "No classes today.")
        else:
            print("No data found for this day (check if it is a weekend).")
        return

    # 4. Fallback: Search for Course Names (e.g., "Basket 5")
    print(f"--- Weekly Schedule for: {query.upper()} ---")
    found_any = False
    for day_idx, row in df_final.iloc[1:].iterrows():
        day_name = str(row['B'])
        day_slots = [(time_map[c], v) for c, v in row.items() if c != 'B' and query in str(v).lower()]
        if day_slots:
            found_any, merged_day = True, merge_consecutive_slots(day_slots)
            print(f"\n{day_name}:")
            for s, e, sub in merged_day: print(f"  └─ {s}-{e}: {sub}")

    if not found_any: print(f"Sorry, I couldn't find any results for '{query}'.")

# --- Run ---
user_input = input("How can I help with your timetable? ")
timetable_assistant(user_input)

How can I help with your timetable?  basket 5 today
--- Schedule for Wednesday, 06 May 2026 ---
• 09:00-10:30: Basket 5
• 10:45-12:15: Data Security and Privacy
• 14:00-15:30: Basket 4
• 15:40-17:10: Basket 3
• 17:30-18:30: Basket 1-T


In [36]:
user_input = input("How can I help with your timetable? ")
timetable_assistant(user_input)


How can I help with your timetable? classes on 8 may 2026
--- Weekly Schedule for: CLASSES ON 8 MAY 2026 ---
Sorry, I couldn't find any results for 'classes on 8 may 2026'.


In [37]:
user_input = input("How can I help with your timetable? ")
timetable_assistant(user_input)


How can I help with your timetable? tommorrow class
--- Weekly Schedule for: TOMMORROW CLASS ---
Sorry, I couldn't find any results for 'tommorrow class'.


In [42]:
user_input = input("How can I help with your timetable? ")
timetable_assistant(user_input)


How can I help with your timetable? 04-05-2026 12:15 class
--- Schedule for Monday, 04 May 2026 ---
• 12:15-13:15: Data Security and Privacy-T


In [47]:
user_input = input("How can I help with your timetable? ")
timetable_assistant(user_input)


How can I help with your timetable? data
--- Weekly Schedule for: DATA ---

MON:
  └─ 12:15-13:15: Data Security and Privacy-T

WED:
  └─ 10:45-12:15: Data Security and Privacy

THU:
  └─ 09:00-10:30: Data Security and Privacy
